# Test submission v4 — `matching_results.tsv` (upload) + `candidate_pairs.tsv`

**Run `er_validation_v4.ipynb` first.** Its last cell writes `er_work/final_v4/` (2 models + metrics + vocab + syllables). Attach that folder, `er_pipeline.py` v3.0, `validate_submission.py`, `new_parquet/indic_dict.json` and the `test/` TSVs as inputs.

Steps: clean test → fillers → partitions → blocking → stage 1 → candidate expansion → stage 2 → decision (expected-F0.5 or threshold, whichever won on validation) → files → official validator. CPU only.

Everything is cached under `er_work/test_run_v4/`. After a disconnect: re-run cells 1–2, then continue from where it stopped.

In [1]:
# ===== CELL 1: install =====
!pip -q install lightgbm rapidfuzz sparse_dot_topn indic_transliteration
import sys, os, json, gc, importlib, time, glob, shutil
import numpy as np, pandas as pd

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 63.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 121.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 309.6/309.6 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.9/162.9 kB 13.9 MB/s eta 0:00:00


In [2]:
# ===== CELL 2: configuration (Google Colab + Drive) =====
from google.colab import drive
drive.mount("/content/drive")
ROOT        = "/content/drive/MyDrive/amazon ml new arch"
CODE_DIR    = ROOT
FINAL_DIR   = f"{ROOT}/er_work/final_v4"
DATA_DIR    = ROOT
INDIC_DICT  = f"{ROOT}/new_parquet/indic_dict.json"
VALIDATOR   = f"{ROOT}/validate_submission.py"
PARQUET_DIR = f"{ROOT}/new_parquet" if os.path.exists(f"{ROOT}/new_parquet/test_source1.parquet") else None

BASE     = ROOT
WORK_DIR = f"{BASE}/er_work"
TEST_DIR = f"{WORK_DIR}/test_run_v4"
OUT_DIR  = f"{BASE}/output"
BLOCK_WORKERS = 6      # 24 cores
FEAT_WORKERS  = 12
os.makedirs(TEST_DIR, exist_ok=True); os.makedirs(OUT_DIR, exist_ok=True)

sys.path.insert(0, CODE_DIR)
import er_pipeline as er
importlib.reload(er)
assert er.VERSION >= "3.0", f"expected er_pipeline.py v3.0, got {er.VERSION}"
MET = json.load(open(f"{FINAL_DIR}/metrics_v4.json"))
DEC, MIN_P, CONF_EXP = MET["decision"], MET["min_p"], MET["conf_exp"]
print("code", CODE_DIR, "| final models", FINAL_DIR, "| data", DATA_DIR, "| parquet", PARQUET_DIR)
print("decision from validation:", DEC, "| min_p", MIN_P, "| conf_exp", CONF_EXP)
print("validation F0.5:", round(MET["final"]["F05"], 6))
!free -h

Mounted at /content/drive
code /content/drive/MyDrive/amazon ml new arch | final models /content/drive/MyDrive/amazon ml new arch/er_work/final_v4 | data /content/drive/MyDrive/amazon ml new arch | parquet /content/drive/MyDrive/amazon ml new arch/new_parquet
decision from validation: {'method': 'expected', 'F05': 0.9870358645282589, 'gamma': 1.0, 'lam': 0.0, 'miss': 0.25, 'threshold_fallback': {'tau': 0.7, 'F05': 0.9867445994201515}} | min_p 0.02 | conf_exp 0.7
validation F0.5: 0.987036
               total        used        free      shared  buff/cache   available
Mem:            47Gi       2.2Gi        33Gi       4.7Mi        12Gi        44Gi
Swap:             0B          0B          0B


In [3]:
# ===== CELL 3: models =====
import lightgbm as lgb
m1 = lgb.Booster(model_file=f"{FINAL_DIR}/model_stage1.txt")
m2 = lgb.Booster(model_file=f"{FINAL_DIR}/model_stage2.txt")
assert m1.num_feature() == len(er.FEATURES_S1) and m2.num_feature() == len(er.FEATURES_S2), "model / er_pipeline version mismatch"
print("models ready:", m1.num_trees(), "and", m2.num_trees(), "trees")

models ready: 1034 and 346 trees


In [4]:

# ===== CELL 4: clean + parse test (all test countries) =====
t0 = time.time()
voc_train = json.load(open(f"{FINAL_DIR}/train_vocab.json"))
er.prepare_split("test", DATA_DIR, WORK_DIR, INDIC_DICT, parquet_dir=PARQUET_DIR, extra_vocab=voc_train)
print(f"{(time.time() - t0) / 60:.1f} min")

[13:17:14] [prepare test] Indic dictionary entries: 1,302 | workers: 24
[13:17:14] [prepare test] S1 cached
[13:17:17] [prepare test] S2 cached
[13:17:17] [prepare test] S3 cached
[13:17:17] [prepare test] done
0.1 min


In [5]:
# ===== CELL 5: fillers (label-free, on test) + all test records + v3 resources =====
fp = f"{TEST_DIR}/fillers_test.json"
if not os.path.exists(fp):
    json.dump(er.detect_fillers("test", WORK_DIR), open(fp, "w"))
fillers_test = json.load(open(fp))
sp = f"{TEST_DIR}/selected.parquet"
if os.path.exists(sp):
    S = pd.read_parquet(sp); parts = json.load(open(f"{TEST_DIR}/partitions.json"))
    for c in ("src", "country"): S[c] = S[c].astype("category")
else:
    S, parts = er.load_selection("test", WORK_DIR, frac=1.0, fillers=fillers_test)
    S.to_parquet(sp, index=False); json.dump(parts, open(f"{TEST_DIR}/partitions.json", "w"))
syl_train = json.load(open(f"{FINAL_DIR}/syllables_train.json"))
S, SYLL = er.setup_v3_resources(S, "test", DATA_DIR, WORK_DIR, PARQUET_DIR, extra_syllables=syl_train)
print(S.groupby(["country", "src"], observed=True).size().unstack())
print("partitions:", len(parts), "| syllables:", len(SYLL))

[13:18:38] [v3 resources test] syllables=126 | split vocab=62,243 | 42s
src          S1       S2       S3
country                          
France   259452   703378   731615
India    809986  2312563  2404998
US       663106  1871330  1945701
partitions: 67 | syllables: 126


In [6]:
# ===== CELL 6: blocking (states in parallel, each saved when done) =====
t0 = time.time()
C = er.run_blocking(S, parts, TEST_DIR, n_workers=BLOCK_WORKERS)
print(f"candidates {len(C):,} | per S1 {len(C) / (S.src == 'S1').sum():.1f} | {(time.time() - t0) / 60:.1f} min")

[13:18:46] [block] 70 jobs cached, 0 to run | workers=6 | kNN threads/job=4
candidates 58,854,242 | per S1 34.0 | 2.3 min


In [7]:
# ===== CELL 7: stage 1 - features + prediction in chunks (only probabilities stored) =====
t0 = time.time()
res = er.build_resources(S, fillers_test)
p1 = er.predict_stage1_streaming(C, S, res, m1, TEST_DIR, n_workers=FEAT_WORKERS)
print(f"stage 1 done | {(time.time() - t0) / 60:.1f} min")

stage 1 done | 0.7 min


In [8]:
# ===== CELL 8: candidate expansion + its stage-1 predictions =====
t0 = time.time()
ep = f"{TEST_DIR}/E.parquet"
if os.path.exists(ep):
    E = pd.read_parquet(ep); p1E = np.load(f"{TEST_DIR}/p1E.npy")
else:
    E = er.expand_candidates(C, S, p1, conf=CONF_EXP)
    XE = er.features_for_new(C, E, S, res, n_workers=FEAT_WORKERS)
    p1E = m1.predict(XE).astype(np.float32); del XE; gc.collect()
    E.to_parquet(ep, index=False); np.save(f"{TEST_DIR}/p1E.npy", p1E)
CE = er.concat_candidates(C, E); nC = len(C)
p1CE = np.r_[p1, p1E].astype(np.float32)
del E; gc.collect()
print(f"expansion rows {len(CE) - nC:,} | {(time.time() - t0) / 60:.1f} min")

expansion rows 4,034,287 | 0.1 min


In [9]:
# ===== CELL 9: stage 2 (siblings) =====
t0 = time.time()
p = er.predict_stage2(CE, S, res, p1CE, m2, TEST_DIR, min_p=MIN_P, n_c=nC, n_workers=FEAT_WORKERS)
print(f"stage 2 done | {(time.time() - t0) / 60:.1f} min")

stage 2 done | 0.1 min


In [ ]:
# ===== CELL 10: decide + write the submission files =====
keep = er.apply_decision(CE, p, DEC, S)
mpath, cpath = er.write_submission(S, CE, keep, OUT_DIR)
s1 = (S.src == "S1").values
has = pd.Series(False, index=np.flatnonzero(s1)); has[np.unique(CE.i1.values[keep])] = True
print("share of S1 with >=1 match, by country (train ~94%):")
print(pd.Series(has.values, index=S.country.astype(str).values[s1]).groupby(level=0).mean().round(3).to_string())
print("matched pairs from expansion:", int(keep[nC:].sum()))
print("\nfiles:", mpath, cpath)

[23:39:41] [submission] S1 rows 1,732,544 | with >=1 match 1,632,573 (94.2%) | matched pairs 5,781,870
share of S1 with >=1 match, by country (train ~94%):
France    0.941
India     0.942
US        0.943
matched pairs from expansion: 34128

files: /content/drive/MyDrive/amazon ml new arch/output/matching_results.tsv /content/drive/MyDrive/amazon ml new arch/output/candidate_pairs.tsv


In [ ]:
# ===== CELL 11: official validator =====
!python3 "{VALIDATOR}" --matching "{mpath}" --candidate "{cpath}" --test-dir "{DATA_DIR}/test"

ML Challenge 2026 — submission validator
  test dir: /content/drive/MyDrive/amazon ml new arch/test
  required S1 entities: 1732544
  matching_results.tsv: 1732544 rows (99971 empty, 1632573 non-empty).
  candidate_pairs.tsv: 1732544 rows (0 empty, 1732544 non-empty).

PASS — no blocking issues found. Safe to submit.


If the validator prints **PASS**, upload `output/matching_results.tsv` to the portal. Keep `candidate_pairs.tsv` for the final zip.

In [ ]:
# ===== CELL 12: prior-shift submission -> output_shift/ =====
# countries absent from training get their own shift (open set, derived from the data)
TRAIN_COUNTRIES = set(er.load_raw("train", 1, DATA_DIR, PARQUET_DIR)["country"].astype(str))
SHIFT_UNSEEN     = -1.0
SHIFT_DEFAULT    = -0.3

def _logit(x):
    x = np.clip(x.astype(np.float64), 1e-6, 1 - 1e-6)
    return np.log(x / (1 - x))

cand_cty = S.country.astype(str).values[CE.i1.values]
shift = np.where(pd.Series(cand_cty).isin(TRAIN_COUNTRIES), SHIFT_DEFAULT, SHIFT_UNSEEN)
p_shift = (1 / (1 + np.exp(-(_logit(p) + shift)))).astype(np.float32)

keep_s = er.apply_decision(CE, p_shift, DEC, S)
OUT_SHIFT = f"{BASE}/output_shift"
mpath_s, cpath_s = er.write_submission(S, CE, keep_s, OUT_SHIFT)
keep_0 = er.apply_decision(CE, p, DEC, S)
for c in np.unique(cand_cty):
    m = cand_cty == c
    print(f"{c:8s} matched pairs {int(keep_0[m].sum()):>10,} -> {int(keep_s[m].sum()):>10,}")
!python3 "{VALIDATOR}" --matching "{mpath_s}" --candidate "{cpath_s}" --test-dir "{DATA_DIR}/test"

In [ ]:
# ===== CELL 13: export contested test pairs for the cross-encoder =====
eid = S.entity_id.values
cand_cty = S.country.astype(str).values[CE.i1.values]
band = np.flatnonzero((p >= 0.01) & (p <= 0.99))
pd.DataFrame({"row": band.astype(np.int64), "s1_id": eid[CE.i1.values[band]],
              "pool_id": eid[CE.i2.values[band]], "p": p[band], "country": cand_cty[band]}
            ).to_parquet(f"{TEST_DIR}/ce_test_pairs.parquet", index=False)
print(f"exported {len(band):,} test pairs -> {TEST_DIR}/ce_test_pairs.parquet")
print(pd.Series(cand_cty[band]).value_counts().to_string())

exported 2,191,300 test pairs -> /content/drive/MyDrive/amazon ml new arch/er_work/test_run_v4/ce_test_pairs.parquet
India     903474
US        786286
France    501540


In [ ]:
####################################################################################################
# [3] test_v4.ipynb -> add at the end (replaces the earlier [C]); run cells 1-9 + this -> output_ce/matching_results.tsv
####################################################################################################
# ===== FINAL CELL: test_v4, after cell 9 -> output_ce/ =====
def _logit(x):
    x = np.clip(np.asarray(x, np.float64), 1e-6, 1 - 1e-6)
    return np.log(x / (1 - x))

# cross-encoders used when BOTH their files are in er_work/ce/ (base from Colab, large from Kaggle)
CE_SOURCES = [("base", "ce_val_scores.parquet", "ce_test_scores.parquet"),
              ("large", "ce_large_val_scores.parquet", "ce_large_test_scores.parquet")]
STACK_COLS = None

def stack_features(i1, i2, p_row, ce_list, all_i2, all_p, src_b, addr_b):
    """Stacker inputs for the scored pairs. ce_list: one probability array per cross-encoder (same rows)."""
    global STACK_COLS
    lp = _logit(p_row); lcs = [_logit(c) for c in ce_list]; lcm = np.mean(lcs, axis=0)
    cols = {"lp": lp, "lcm": lcm, "lb": 0.5 * (lp + lcm)}
    for k, lc in enumerate(lcs):
        cols[f"lc{k}"] = lc
    d = pd.DataFrame({"i1": i1, **cols})
    F = dict(cols); F["diff"] = lcm - lp
    g = d.groupby("i1", sort=False)
    F["n_cand"] = g.lp.transform("size").values
    for c in cols:
        r = g[c].rank(ascending=False, method="first").values
        mx = g[c].transform("max").values
        F[f"{c}_rank"] = r; F[f"{c}_gap"] = mx - d[c].values; F[f"{c}_max"] = mx
        F[f"{c}_npos"] = (d[c] > 0).groupby(d.i1, sort=False).transform("sum").values
        F[f"{c}_second"] = d[c].where(r == 2).groupby(d.i1, sort=False).transform("max").fillna(-14).values
    keep = np.isin(all_i2, np.unique(i2)) & (all_p >= 0.001)
    a = pd.DataFrame({"i2": all_i2[keep], "p": all_p[keep]}).sort_values(["i2", "p"], ascending=[True, False])
    rk = a.groupby("i2", sort=False).cumcount().values
    best = a[rk == 0].set_index("i2").p; second = a[rk == 1].set_index("i2").p
    nn = a.groupby("i2", sort=False).size(); s_i2 = pd.Series(i2)
    b1 = s_i2.map(best).fillna(0).values; b2 = s_i2.map(second).fillna(0).values
    other = np.where(p_row >= b1, b2, b1)
    F["pool_best_other"] = other; F["pool_margin"] = p_row - other
    F["pool_is_top"] = (p_row >= b1).astype(np.float32); F["pool_n"] = s_i2.map(nn).fillna(1).values
    F["b_s3"] = src_b.astype(np.float32); F["b_has_addr"] = addr_b.astype(np.float32)
    STACK_COLS = list(F)
    return np.column_stack([np.asarray(F[k], np.float32) for k in STACK_COLS])

STACK_PARAMS = dict(objective="binary", learning_rate=0.05, num_leaves=31, min_data_in_leaf=100,
                    feature_fraction=0.9, bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0, verbose=-1, seed=42)

import lightgbm as lgb
# countries absent from training get their own shift (open set, derived from the data)
TRAIN_COUNTRIES = set(er.load_raw("train", 1, DATA_DIR, PARQUET_DIR)["country"].astype(str))
SHIFT_UNSEEN     = -3.0
SHIFT_DEFAULT    = -0.3
USE_UNSEEN_FT    = True                  # use the self-trained scores for unseen-country pairs, if that file exists
CE_DIR = f"{WORK_DIR}/ce"
BP = json.load(open(f"{CE_DIR}/blend_params.json"))
srcs = [(n, t) for n, v, t in CE_SOURCES if n in BP["ce_used"]]
cts = [pd.read_parquet(f"{CE_DIR}/{t}").set_index("row").ce for _, t in srcs]
r = np.array(sorted(set.intersection(*[set(c.index) for c in cts])), dtype=np.int64)
assert len(r) and r.max() < len(p), "cross-encoder test rows do not match the candidates - wrong files?"
ce_list = [c.reindex(r).values for c in cts]
fr_fp = f"{CE_DIR}/ce_fr_test_scores.parquet"
if USE_UNSEEN_FT and os.path.exists(fr_fp) and "base" in [n for n, _ in srcs]:             # unseen countries: replace the base cross-encoder's score
    fr = pd.read_parquet(fr_fp).set_index("row").ce
    k = [n for n, _ in srcs].index("base")
    ce_list[k] = pd.Series(ce_list[k], index=r).where(~pd.Index(r).isin(fr.index), fr.reindex(r)).values
    print(f"unseen-country self-trained scores used for {int(pd.Index(r).isin(fr.index).sum()):,} pairs")
lg = _logit(p)
if BP["model"] == "stacker":
    i1t, i2t = CE.i1.values, CE.i2.values
    Xt = stack_features(i1t[r], i2t[r], p[r], ce_list, i2t, p,
                        (S.src == "S3").values[i2t[r]], S.has_addr.values.astype(bool)[i2t[r]])
    lg[r] = _logit(lgb.Booster(model_file=f"{CE_DIR}/stacker.txt").predict(Xt))
else:
    w = BP["w"]; lg[r] = w * lg[r] + (1 - w) * np.mean([_logit(c) for c in ce_list], axis=0)
cand_cty = S.country.astype(str).values[CE.i1.values]
lg += np.where(pd.Series(cand_cty).isin(TRAIN_COUNTRIES), SHIFT_DEFAULT, SHIFT_UNSEEN)
p_ce = (1 / (1 + np.exp(-lg))).astype(np.float32)
keep_ce = er.apply_decision(CE, p_ce, BP["decision"], S)
mpath_c, cpath_c = er.write_submission(S, CE, keep_ce, f"{BASE}/output_ce")
keep_0 = er.apply_decision(CE, p, DEC, S)
print("using", BP)
for c in np.unique(cand_cty):
    m_ = cand_cty == c
    print(f"  {c:8s} matched pairs {int(keep_0[m_].sum()):>10,} -> {int(keep_ce[m_].sum()):>10,}")
!python3 "{VALIDATOR}" --matching "{mpath_c}" --candidate "{cpath_c}" --test-dir "{DATA_DIR}/test"